In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
MERGE INTO silver.highscores_summary AS target
USING (
  SELECT
    raw_highscores_id AS snapshot_id,
    sha1(lower(trim(highscores.world))) AS world_id,
    highscores.world AS world,
    highscores.category AS category,
    highscores.vocation AS vocation,
    highscores.highscore_age AS highscore_age,
    highscores.highscore_page.current_page AS current_page,
    highscores.highscore_page.total_pages AS total_pages,
    highscores.highscore_page.total_records AS total_records,
    information.api.version AS api_version,
    information.api.release AS api_release,
    information.api.commit AS api_commit,
    to_timestamp(information.timestamp) AS api_processed_at,
    information.tibia_urls AS tibia_urls,
    information.status.error AS error,
    information.status.http_code AS http_code,
    information.status.message AS message,
    source_file_date,
    source_file_path,
    source_file_modified_at,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_highscores
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
MERGE INTO silver.highscores_characters AS target 
USING (
  SELECT
    raw_highscores_id AS snapshot_id,
    sha1(lower(trim(highscores.world))) AS world_id,
    highscores.world AS world,
    highscores.category AS category,
    highscores.vocation AS vocation,
    character.rank AS rank,
    sha1(concat_ws('|', lower(trim(character.name)), lower(trim(character.world)), source_file_date)) AS snapshot_character_id,
    character.name AS character_name,
    character.title AS character_title,
    character.vocation AS character_vocation,
    character.world AS character_world,
    character.level AS character_level,
    character.value AS character_value,
    source_file_date,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_highscores
  LATERAL VIEW explode(highscores.highscore_list) highscores_table AS character
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;